# Bipartite AML Graph Neural Network Tutorial

This notebook demonstrates how to use the `bipartite_aml_gnn.py` script modules to build, train, and evaluate a Graph Neural Network for Anti-Money Laundering (AML) detection in a bipartite financial network (Customer <-> Bank Account).

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add the scripts directory to the system path to import the module
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'scripts')))

from bipartite_aml_gnn import prepare_hetero_data, UnifiedAMLGNN, train, inference

## 1. Load or Generate Data

First, we'll generate some mock data to simulate financial transactions. In a real-world scenario, you would load your dataset using `pd.read_csv()`.

In [ ]:
# Generate Mock Data
num_tx = 1000
mock_data = {
    'transaction_id': range(num_tx),
    'customer_name': [f'Cust_{{np.random.randint(0, 100)}}' for _ in range(num_tx)],
    'bank_account_id': [f'Acct_{{np.random.randint(0, 50)}}' for _ in range(num_tx)],
    'amount': np.random.rand(num_tx) * 10000,
    'hour_of_day': np.random.randint(0, 24, num_tx),
    # Customer Features
    'cust_risk_score': np.random.rand(num_tx),
    'cust_tenure': np.random.randint(1, 365, num_tx),
    # Bank Features
    'bank_volume': np.random.rand(num_tx) * 100000,
    'bank_flags': np.random.randint(0, 2, num_tx),
    # Labels (0: Normal, 1: Laundering)
    'is_laundering': np.random.choice([0, 1], num_tx, p=[0.95, 0.05])
}
df = pd.DataFrame(mock_data)

# Ensure label consistency per customer
cust_labels = df.groupby('customer_name')['is_laundering'].max()
df['is_laundering'] = df['customer_name'].map(cust_labels)

print(f"Generated {len(df)} transactions.")
df.head()

## 2. Configure Columns

Define which columns correspond to IDs and features.

In [ ]:
CUSTOMER_ID = 'customer_name'
BANK_ID = 'bank_account_id'

CUSTOMER_FEATURES = ['cust_risk_score', 'cust_tenure']
BANK_FEATURES = ['bank_volume', 'bank_flags']
EDGE_FEATURES = ['amount', 'hour_of_day']
LABEL_COL = 'is_laundering'

## 3. Prepare Graph Data

Convert the DataFrame into a PyTorch Geometric `HeteroData` object.

In [ ]:
hetero_data, cust_mapping = prepare_hetero_data(
    df, CUSTOMER_ID, BANK_ID, 
    CUSTOMER_FEATURES, BANK_FEATURES, EDGE_FEATURES, 
    LABEL_COL
)

print(hetero_data)

## 4. Initialize & Train Model

We can choose between 'sage', 'gat', or 'rgcn' architectures.

In [ ]:
MODEL_CHOICE = 'sage'  # Try 'gat' or 'rgcn' too!

model = UnifiedAMLGNN(
    hidden_channels=64, 
    out_channels=1, 
    num_layers=2, 
    architecture=MODEL_CHOICE
)

print("Starting Training...")
model = train(model, hetero_data, epochs=10, lr=0.01)

## 5. Inference & Results

Generate risk scores for all customers.

In [ ]:
risk_scores = inference(model, hetero_data)

# Map integer indices back to original Customer IDs
inv_cust_map = {v: k for k, v in cust_mapping.items()}
risk_scores['customer_id'] = risk_scores['cust_idx'].map(inv_cust_map)

# Display high-risk customers
print("Top 10 High Risk Customers:")
risk_scores.sort_values('risk_score', ascending=False).head(10)